# Version 3 — Existing Pinecone Knowledge Base + PostgreSQL

This version connects to the Pinecone index shown in your screenshots (`ecom-kb`) and searches its three namespaces: `nlq-schema`, `nlq-examples`, and `common`. It does not rebuild or upload the knowledge base.

**Before running:** set `PINECONE_API_KEY`, `OPENAI_API_KEY`, `BEDROCK_CHAT_MODEL_ID`, and PostgreSQL environment variables. The query embedding model and dimension must match the model used when vectors were upserted to Pinecone.


## End-to-end agent workflow — Version 3

```text
User Question
     |
     v
AI Data Analyst Agent
     |
     +---- lookup_schema(term) ---------> Pinecone: nlq-schema
     |
     +---- lookup_examples(question) ---> Pinecone: nlq-examples
     |
     +---- lookup_common(question) -----> Pinecone: common
     |
     v
Generate read-only SQL
     |
     v
run_query(sql) -------------------------> PostgreSQL
     |                                      |
     +<------------- query results --------+
     |
     v
Final Answer grounded in query results
```

**Tools available:** Three Pinecone retrieval tools and `run_query(sql)`. Dedicated SQL validation is introduced in Version 4.

# Version 3 — Pinecone Semantic Memory + PostgreSQL

In Version 2, our agent had a **brain and hands**:
- **Brain:** the language model.
- **Hands:** `run_query(sql)`, which executes read-only SQL against PostgreSQL.

In Version 3, we add a **knowledge lookup tool** called `lookup_schema(term)`.

This tool searches a semantic memory containing schema knowledge: table descriptions, column definitions, synonyms, and join guidance. The agent can retrieve the relevant schema before writing SQL, instead of guessing column names.

**Scope:** This version adapts only the `lookup_schema` capability from `nlq.zip`. It does not add the other tools from that project.

## 1. The new architecture

```text
User question
     |
     v
Agent + model (brain)
     |
     +---- lookup_schema(term) ----> Semantic schema memory
     |                                  |
     |<------- relevant schema ----------+
     |
     +---- run_query(sql) ----------> PostgreSQL
     |                                  |
     |<------- query results ------------+
     |
     v
Final answer grounded in query results
```

**Why semantic memory?** Users may ask about “buyers,” while the schema calls them `customers`; or ask for “money kept,” while the business definition uses payment status. Embedding-based retrieval can find relevant schema descriptions even when the wording differs.

## 2. Install dependencies

We use:
- `langchain-openai` for the model and embeddings.
- `langchain` for the agent and tools.
- `pandas` and `openpyxl` to read the knowledge-base workbook.
- `numpy` for cosine similarity.
- `python-dotenv` for environment variables.
- `psycopg` for PostgreSQL.

In [ ]:
%pip install -q langchain langchain-aws langchain-openai pinecone requests python-dotenv "psycopg[binary]"


## 3. Configure the model and database

Place `kb_feed_ecom.xlsx` in the same folder as this notebook, or update `KB_PATH`. Configure the API key and PostgreSQL connection in environment variables or a `.env` file.

Required environment variables:
- `OPENAI_API_KEY`
- `PGHOST`
- `PGDATABASE`
- `PGUSER`
- `PGPASSWORD`

Optional:
- `PGPORT` (defaults to `5432`)
- `PGSSLMODE` (set to the value required by your database provider)

In [ ]:
import os
import json
import psycopg
from psycopg.rows import dict_row
from dotenv import load_dotenv

from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_aws import ChatBedrockConverse
from langchain_openai import OpenAIEmbeddings
from pinecone import Pinecone

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY", "")
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "ecom-kb")
EMBEDDING_MODEL = os.getenv("PINECONE_EMBEDDING_MODEL", "text-embedding-3-small")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
BEDROCK_CHAT_MODEL_ID = os.getenv("BEDROCK_CHAT_MODEL_ID", "")

if not PINECONE_API_KEY:
    raise ValueError("Set PINECONE_API_KEY.")
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY for query embeddings. It must match the embedding model used to build the Pinecone index.")
if not BEDROCK_CHAT_MODEL_ID:
    raise ValueError("Set BEDROCK_CHAT_MODEL_ID to a chat model enabled in your AWS account.")

required_pg_vars = ["PGHOST", "PGDATABASE", "PGUSER", "PGPASSWORD"]
missing_pg_vars = [name for name in required_pg_vars if not os.getenv(name)]
if missing_pg_vars:
    raise ValueError("Missing PostgreSQL environment variables: " + ", ".join(missing_pg_vars))

PGPORT = int(os.getenv("PGPORT", "5432"))
PGSSLMODE = os.getenv("PGSSLMODE")

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
pinecone = Pinecone(api_key=PINECONE_API_KEY)
index = pinecone.Index(PINECONE_INDEX_NAME)
model = ChatBedrockConverse(
    model=BEDROCK_CHAT_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0,
)


## 4. Build the semantic schema memory

We use the workbook's `nlq_table` and `nlq_column` sheets as the source of truth for schema knowledge.

Each active table or column becomes a searchable document. We embed those documents once, then use cosine similarity to retrieve the most relevant records for a lookup.

This is a small, in-notebook semantic index for teaching. It is not a persistent vector database. For production, the same retrieval pattern can be backed by Pinecone or another vector store.

## 5. Connect to the existing Pinecone knowledge base

Important: the workbook has a separate `nlq_examples` sheet. Schema lookup alone cannot retrieve worked SQL examples. We load those questions, paraphrases, and verified SQL statements into a second semantic index so the agent can find a similar example before drafting a query.


## 5. Embed the schema documents

The embedding model converts schema descriptions into vectors. The lookup tool embeds the user's search phrase and compares it with these vectors.

For a large knowledge base, store the vectors in a persistent vector index instead of recomputing them every time the notebook starts.

## 6. Add the `lookup_schema(term)` tool

This is the new **knowledge lookup tool**. It does not query PostgreSQL. It searches schema memory and returns relevant table and column definitions to the agent.

Like the schema lookup tool in `nlq.zip`, it:
- Searches for the schema concept requested by the agent.
- Returns relevant table/column descriptions.
- Records concepts already searched during the current user turn, so repeated rephrasings do not trigger repeated retrieval calls.

The repeat cache below is deliberately scoped to a single `ask_agent()` call for this notebook demonstration.

## 7. Keep the PostgreSQL tool: `run_query(sql)`

This remains the agent's **hands** for database execution. The new `lookup_schema` tool is its **semantic memory**. They have different responsibilities:
- `lookup_schema` retrieves knowledge about the schema.
- `run_query` executes SQL and returns database results.

## 8. Add `lookup_examples(question)`

`lookup_examples` searches the existing Pinecone examples index. `lookup_schema` answers **which tables and columns mean what**; `lookup_examples` answers **has a similar question already been solved, and what SQL pattern was used?**


## Pinecone hybrid retrieval: vector kNN + BM25

The supplied `nlq.zip` code uses Pinecone indexes, Bedrock Titan embeddings, and AWS SigV4 authentication. A query is sent to both the vector field (`embedding`) and text field (`text`), then the index returns matching records.

**Important:** the supplied `nlq.zip` implementation uses Pinecone, while your screenshots show Pinecone. This notebook follows the live Pinecone index shown in your screenshots.


In [ ]:
@tool
def run_query(sql: str) -> str:
    """Execute one read-only SELECT query against the configured PostgreSQL database.

    Use only after consulting lookup_schema when table/column details are not
    already established. Returns at most 50 rows.
    """
    sql_clean = sql.strip()
    if not sql_clean:
        return "REFUSED: SQL query is empty."

    first_word = sql_clean.split(None, 1)[0].upper()
    if first_word not in {"SELECT", "WITH"}:
        return "REFUSED: Only read-only SELECT queries are allowed."

    connection_kwargs = {
        "host": os.environ["PGHOST"],
        "port": PGPORT,
        "dbname": os.environ["PGDATABASE"],
        "user": os.environ["PGUSER"],
        "password": os.environ["PGPASSWORD"],
        "connect_timeout": 10,
        "row_factory": dict_row,
    }
    if PGSSLMODE:
        connection_kwargs["sslmode"] = PGSSLMODE

    try:
        with psycopg.connect(**connection_kwargs) as conn:
            with conn.transaction():
                conn.execute("SET TRANSACTION READ ONLY")
                conn.execute("SET LOCAL statement_timeout = '15000ms'")
                cursor = conn.execute(sql_clean)
                rows = cursor.fetchmany(50)
                columns = [item.name for item in cursor.description] if cursor.description else []

        if not rows:
            return "Query succeeded. 0 rows returned."

        header = " | ".join(columns)
        body = "\n".join(
            " | ".join(str(row.get(column)) for column in columns)
            for row in rows
        )
        return f"{header}\n{body}\n(Returned at most 50 rows.)"

    except (psycopg.Error, ValueError) as exc:
        return f"SQL error: {exc}"

In [ ]:
def search_namespace(namespace: str, question: str, top_k: int = 8) -> list[dict]:
    """Embed a question and search the selected existing Pinecone namespace."""
    vector = embeddings.embed_query(question)
    response = index.query(
        vector=vector,
        namespace=namespace,
        top_k=top_k,
        include_metadata=True,
    )
    matches = response.get("matches", []) if isinstance(response, dict) else response.matches
    results = []
    for match in matches:
        if isinstance(match, dict):
            match_id = match.get("id", "unknown")
            score = match.get("score")
            metadata = match.get("metadata") or {}
        else:
            match_id = getattr(match, "id", "unknown")
            score = getattr(match, "score", None)
            metadata = getattr(match, "metadata", None) or {}
        results.append({"id": match_id, "score": score, "metadata": metadata})
    return results


def format_matches(matches: list[dict], max_chars: int = 12000) -> str:
    """Format Pinecone metadata while omitting vector payloads."""
    if not matches:
        return "No matching knowledge-base records were found."
    blocks = []
    for rank, match in enumerate(matches, start=1):
        lines = [
            f"RESULT {rank} | id={match['id']} | score={match['score']}"
        ]
        for key, value in match["metadata"].items():
            if isinstance(value, (dict, list)):
                value = json.dumps(value, ensure_ascii=False)
            lines.append(f"{key}: {value}")
        blocks.append("\n".join(lines))
    output = "\n\n---\n\n".join(blocks)
    return output[:max_chars] + ("\n...[truncated]" if len(output) > max_chars else "")


@tool
def lookup_schema(term: str) -> str:
    """Search the nlq-schema Pinecone namespace for tables and columns.
    Use it to confirm real table/column names, data types, descriptions, synonyms,
    and business meaning. Retrieves metadata only; does not execute SQL."""
    return format_matches(search_namespace("nlq-schema", term, top_k=8))


@tool
def lookup_examples(question: str) -> str:
    """Search the nlq-examples Pinecone namespace for similar approved NL-to-SQL examples.
    Use examples as patterns, then verify schema details using lookup_schema.
    Retrieves examples only; does not execute SQL."""
    return format_matches(search_namespace("nlq-examples", question, top_k=4), max_chars=10000)


@tool
def lookup_common(question: str) -> str:
    """Search the common Pinecone namespace for shared business definitions or guidance.
    Retrieves knowledge only; does not execute SQL."""
    return format_matches(search_namespace("common", question, top_k=4), max_chars=8000)


## 8. Update the system prompt

The agent now has four tools: schema lookup, SQL-example lookup, common-knowledge lookup, and database query execution. We explicitly explain when to use each one and tell the agent to consult semantic schema memory before guessing column names.

In [ ]:
SYSTEM_PROMPT = """
ROLE
You are an AI Data Analyst Agent for an e-commerce marketplace.

CONTEXT
The existing Pinecone index `ecom-kb` contains three knowledge namespaces:
- `nlq-schema`: schema metadata, columns, relationships, and related descriptions.
- `nlq-examples`: similar natural-language questions and SQL examples.
- `common`: shared business definitions and guidance.
PostgreSQL is the source of truth for internal marketplace data.

AVAILABLE TOOLS
- lookup_schema(term): searches Pinecone namespace `nlq-schema`.
- lookup_examples(question): searches Pinecone namespace `nlq-examples`.
- lookup_common(question): searches Pinecone namespace `common`.
- run_query(sql): executes a read-only SQL query against PostgreSQL and returns up
  to 50 rows.
This version does not have a dedicated SQL validation tool.

TOOL-SELECTION AND WORKFLOW
- Use lookup_schema when table/column names, joins, grain, or definitions are not
  already established by reliable context.
- Use lookup_examples when a similar approved question or SQL pattern may help.
- Use lookup_common when shared business definitions or guidance may affect the answer.
- Do not call every retrieval tool automatically; choose those relevant to the question.
- Treat retrieved examples as patterns, not guaranteed-correct answers. Verify columns,
  joins, filters, date ranges, and aggregation against retrieved schema.
- Generate one read-only SELECT/WITH query for internal data questions, then call
  run_query(sql). Inspect the result before answering.
- If SQL fails, use the error and retrieved knowledge to correct it when possible.
- Do not claim a query succeeded unless run_query confirms it.
- If schema details are insufficient, say what is missing rather than inventing them.

CONSTRAINTS
- Never invent numbers, results, schema details, or business definitions.
- Consider table grain, join cardinality, NULLs, and double-counting.
- For percentages, state the numerator and denominator.
- If evidence is insufficient, state UNKNOWN and explain why.
- Do not expose private chain-of-thought; provide a concise explanation only.
- Read-only restrictions must also be enforced by database permissions.

OUTPUT FORMAT
Use relevant sections:
ANSWER: Answer with units where applicable, or UNKNOWN.
DENOMINATOR: For percentages, identify numerator and denominator; otherwise N/A.
SOURCE: Identify PostgreSQL and include the SQL actually executed, if successful.
CAVEAT: State material limitations, ambiguity, or data-quality concerns.
"""

## 9. Create the agent with both tools

Version 2 had one tool. Version 3 now passes both tools to `create_agent()`.

In [ ]:
agent = create_agent(
    model=model,
    tools=[lookup_schema, lookup_examples, lookup_common, run_query],
    system_prompt=SYSTEM_PROMPT,
)

## 10. Run a question and inspect the tool loop

The trace shows whether the agent first retrieves schema knowledge, then executes SQL, and finally answers based on the returned result.

In [ ]:
# Test retrieval tools independently before asking the agent a full question.
print(lookup_schema.invoke({"term": "product category identifier"}))
print("\n--- Similar SQL examples ---\n")
print(lookup_examples.invoke({"question": "top products by units sold"}))
print("\n--- Common knowledge ---\n")
print(lookup_common.invoke({"question": "how should revenue be calculated?"}))


## 11. Test schema lookup on its own

You can call the tool directly to see what semantic memory retrieves. This is only a retrieval test; it does not execute SQL.

In [ ]:
# Test retrieval tools independently before asking the agent a full question.
print(lookup_schema.invoke({"term": "product category identifier"}))
print("\n--- Similar SQL examples ---\n")
print(lookup_examples.invoke({"question": "top products by units sold"}))
print("\n--- Common knowledge ---\n")
print(lookup_common.invoke({"question": "how should revenue be calculated?"}))


## 12. Teaching narration: from tools to semantic memory

> “In Version 2, our agent had a brain and hands. It could query PostgreSQL, but it still needed reliable knowledge about table names, column names, joins, and business meanings.
>
> In Version 3, we add semantic memory through `lookup_schema`. Before writing SQL, the agent can search the knowledge base using the meaning of the question, not just an exact keyword match.
>
> The lookup tool retrieves schema knowledge. The query tool retrieves live data. One tells the agent **how the data is structured**; the other tells it **what the data contains**.”

| Tool | Purpose | Analogy |
|---|---|---|
| `lookup_schema(term)` | Retrieves schema knowledge by semantic similarity | **Memory / reference library** |
| `run_query(sql)` | Executes SQL against PostgreSQL | **Hands / action** |
| GPT-6 model | Chooses which tool to call and interprets results | **Brain / decision-maker** |

## 13. Important limitations

- Keep `kb_feed_ecom.xlsx` beside the notebook, or change `KB_PATH`.
- This example uses in-memory embeddings, so the index is rebuilt when the notebook starts.
- The workbook's examples may use a different SQL dialect; use schema and relationship knowledge, but ensure generated SQL matches PostgreSQL.
- The similarity threshold (`0.30`) and top-k (`8`) are starting values, not universal settings; evaluate them against representative questions.
- The database role should have read-only permissions. Basic SQL prefix checks are not a complete security validator.
- The model and embedding API must both be accessible through your API account.

**Next step:** Evaluate retrieval quality with a set of questions whose relevant tables and columns are known, then tune top-k and the similarity threshold.